# MA(q) synthetic series vs TimesFM 2.5

**Goal:** Compare one-step-ahead forecasts from a fitted **MA(q)** model to **TimesFM** on synthetic **MA(q)** data.

**Setting (same as the AR notebook):**
- **Horizon:** $t{+}1$ only.
- **Window:** History length 11 first ($y_0,\ldots,y_{10}$), then expanding; test indices $k = 11,\ldots,99$ (**89** points) with $n=100$.
- **Metric:** MSE of one-step errors.
- **DGP:** Gaussian **MA(q)** with **q ∈ {2, 5}**, **n = 100**. Classical order matches the DGP: **ARIMA(0,0,q)**.

**Interpretation:** The MA model is *estimated* on each expanding history; TimesFM gets the same history as its input prompt.

## 1. Setup

**Hugging Face:** set `HF_TOKEN` ([tokens](https://huggingface.co/settings/tokens)). From repo root: `pip install -e ".[experiments]"`, copy `.env.example` to `.env`, add `HF_TOKEN=...`, or `export HF_TOKEN=...` before Jupyter.

Imports, repo path, `load_dotenv`, fixed **random seed**.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="statsmodels")

HERE = Path.cwd().resolve()
REPO = HERE if (HERE / "src" / "timesfm").is_dir() else HERE.parent
try:
    from dotenv import load_dotenv
    load_dotenv(REPO / ".env")
except ImportError:
    pass
sys.path.insert(0, str(REPO / "src"))

import timesfm
from statsmodels.tsa.arima.model import ARIMA

RNG_SEED = 42
N = 100
K_FIRST = 11
rng = np.random.default_rng(RNG_SEED)

/Users/nikhileshbelulkar/Documents/timesfm_google/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Simulate MA(q) data

Gaussian **MA(q)** in standard form:

$y_t = \varepsilon_t + \theta_1 \varepsilon_{t-1} + \cdots + \theta_q \varepsilon_{t-q}$,

with i.i.d. standard normal $\varepsilon$. We draw a long enough white-noise path and build $y_t$ in a plain loop. Coefficients are fixed (invertible / stable).

In [2]:
def simulate_ma(q: int, n: int, rng: np.random.Generator) -> np.ndarray:
    if q == 2:
        theta = np.array([0.6, 0.3])
    elif q == 5:
        theta = np.array([0.25, 0.20, 0.15, 0.12, 0.10])
    else:
        raise ValueError("q must be 2 or 5")
    # Shocks e_0, e_1, ... long enough to form y_0,...,y_{n-1}
    e = rng.normal(0.0, 1.0, n + q)
    y = np.zeros(n, dtype=np.float64)
    for t in range(n):
        # e_t is e[t+q] so indices t..t+q line up with y_t using e_{t},...,e_{t-q}
        term = e[t + q]
        for j in range(1, q + 1):
            term += theta[j - 1] * e[t + q - j]
        y[t] = term
    return y


y_ma2 = simulate_ma(2, N, rng)
y_ma5 = simulate_ma(5, N, rng)

## 3. One-step MA forecast (classical)

Fit **ARIMA(0,0,q)** on `history` and return the one-step forecast.

In [3]:
def forecast_ma(history: np.ndarray, q: int) -> float | None:
    try:
        if len(history) <= q:
            return None
        fit = ARIMA(history.astype(np.float64), order=(0, 0, q)).fit()
        return float(np.asarray(fit.forecast(steps=1)).ravel()[0])
    except Exception:
        raise Exception

## 4. Load TimesFM (once)

**horizon = 1** when we evaluate below.

In [4]:
torch.set_float32_matmul_precision("high")
tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch",
    torch_compile=False,
)
tfm.compile(
    timesfm.ForecastConfig(
        max_context=1024,
        max_horizon=256,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=False,
        fix_quantile_crossing=True,
    )
)

## 5. Expanding-window loop and MSE

For each DGP order **q**, test $k = 11,\ldots,99$: actual $= y[k]$, history $= y[:k]$. Compare **MA(q)** vs **TimesFM**.

In [5]:
rows: list[dict] = []
test_ks = list(range(K_FIRST, N))

for q, y in [(2, y_ma2), (5, y_ma5)]:
    se_ma: list[float] = []
    se_tf: list[float] = []
    n_fail = 0
    for k in test_ks:
        actual = float(y[k])
        hist = y[:k].astype(np.float32)
        pred_tf = float(tfm.forecast(horizon=1, inputs=[hist])[0][0, 0])
        pred_ma = forecast_ma(y[:k], q)
        e_ma = actual - pred_ma
        e_tf = actual - pred_tf
        se_ma.append(e_ma**2)
        se_tf.append(e_tf**2)
    rows.append(
        {
            "q_dgp": q,
            "n_test": len(test_ks),
            "ma_fit_failures": n_fail,
            "mse_ma": float(np.mean(se_ma)) if se_ma else float("nan"),
            "mse_timesfm": float(np.mean(se_tf)) if se_tf else float("nan"),
        }
    )

results = pd.DataFrame(rows)
results

/Users/nikhileshbelulkar/Documents/timesfm_google/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/nikhileshbelulkar/Documents/timesfm_google/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/nikhileshbelulkar/Documents/timesfm_google/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/nikhileshbelulkar/Documents/timesfm_google/.venv/lib/python3.11/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_r

,q_dgp,n_test,ma_fit_failures,mse_ma,mse_timesfm
0,2,89,0,0.623531,0.679893
1,5,89,0,1.182167,1.089988


## 6. Save table

CSV under `output/`.

In [6]:
out_dir = HERE / "output"
out_dir.mkdir(parents=True, exist_ok=True)
path = out_dir / "ma_vs_timesfm_results.csv"
results.to_csv(path, index=False)
print(path)

/Users/nikhileshbelulkar/Documents/timesfm_google/experiments_nikhilesh_experiments/output/ma_vs_timesfm_results.csv
